In [1]:
from dotenv import load_dotenv,find_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os

In [2]:
load_dotenv(find_dotenv())

True

In [3]:
openrouter_key = os.getenv("OPENROUTER_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY");

In [4]:
client_openrouter = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=openrouter_key
)

In [5]:
client_gemini = OpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [6]:
reader = PdfReader("./me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    linkedin += text

In [7]:
print(linkedin)

   
Contact
7036086060 (Home)
kondajayanthreddy@gmail.com
www.linkedin.com/in/
jayanthreddykonda (LinkedIn)
Top Skills
Machine Learning Algorithms
React.js
Data Structures
Certifications
Machine Learning by Stanford
University & DeepLearning.AI on
Coursera
Jayanth Reddy Konda
AI & Machine Learning Undergraduate | Laying Strong Foundations
in Intelligent, Data-Driven Systems
Hyderabad, Telangana, India
Summary
I'm an AI & Machine Learning undergraduate who enjoys turning
curiosity into code and data into insight. Currently pursuing my
B.Tech in Computer Science (AIML) at VNR Vignana Jyothi Institute
of Engineering and Technology, I'm focused on building strong
foundations in intelligent, data-driven systems-the kind designed to
work beyond theory and textbooks.
I'm actively exploring machine learning concepts, data analysis,
algorithms, and problem-solving techniques. I value understanding
why something works before jumping into how, and I enjoy breaking
complex problems into manageable

In [8]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [9]:
name = "Jayanth Reddy Konda"

In [10]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [11]:
print(system_prompt)

You are acting as Jayanth Reddy Konda. You are answering questions on Jayanth Reddy Konda's website, particularly questions related to Jayanth Reddy Konda's career, background, skills and experience. Your responsibility is to represent Jayanth Reddy Konda for interactions on the website as faithfully as possible. You are given a summary of Jayanth Reddy Konda's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.

## Summary:
I'm Jayanth an AI & Machine Learning undergraduate who enjoys turning
curiosity into code and data into insight. Currently pursuing my
B.Tech in Computer Science (AIML) at VNR Vignana Jyothi Institute
of Engineering and Technology, I'm focused on building strong
foundations in intelligent, data-driven systems-the kind designed to
work beyond theory and textbooks

## LinkedIn Profile:
   
Contact
7

In [12]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client_openrouter.chat.completions.create(model="nvidia/nemotron-3-super-120b-a12b:free", messages=messages)
    return response.choices[0].message.content

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [13]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [14]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [15]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [16]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = client_gemini.chat.completions.parse(model="gemini-3-flash-preview", messages=messages, response_format=Evaluation)
    parsed = response.choices[0].message.parsed
    if parsed is None:
        raise ValueError("Gemini did not return a valid evaluation.")
    return parsed

In [17]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = client_openrouter.chat.completions.create(model="nvidia/nemotron-3-super-120b-a12b:free", messages=messages)
    return response.choices[0].message.content

In [18]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = client_openrouter.chat.completions.create(model="nvidia/nemotron-3-super-120b-a12b:free", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat).launch(prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The agent responded using Pig Latin, which is unprofessional and potentially confusing for a user. The agent is supposed to represent Jayanth Reddy Konda to potential employers or clients and should maintain a professional tone in plain English.
